# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## My Rule and Its Reason Codes

### Rule

I will prioritize content that is both **stale** and **visible**.

A page is considered stale when it has not been updated for at least **180 days**. A page is considered visible when it has at least **500 impressions in the last 90 days**.

Pages that satisfy both conditions receive a score equal to their 90-day impressions. This gives higher priority to stale pages that still have meaningful search exposure.

### Reason codes

- `stale_visible`: The page is stale and has meaningful search exposure, so it is a candidate for refresh.
- `not_priority`: The page does not satisfy both baseline conditions.

### Action labels

- `refresh_first`: Prioritize the page for editorial review.
- `monitor`: Do not prioritize it under this baseline rule.

This is a directional decision-support rule, not a guarantee that refreshing a page will improve its performance.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:", len(df.columns))

# Signal 1: staleness
df["stale"] = df["days_since_last_update"] >= 180

# Signal 2: visibility
df["visible"] = df["impressions_90d"] >= 500

# Signal check 1: staleness buckets
stale_check = (
    df["stale"]
    .value_counts()
    .rename_axis("stale")
    .reset_index(name="n")
)

print("Staleness signal")
print(stale_check)

# Signal check 2: visibility buckets
visible_check = (
    df["visible"]
    .value_counts()
    .rename_axis("visible")
    .reset_index(name="n")
)

print("\nVisibility signal")
print(visible_check)

# Rule
df["baseline_score"] = (
    df["stale"].astype(int)
    * df["visible"].astype(int)
    * df["impressions_90d"]
)

df["reason_code"] = np.where(
    (df["stale"]) & (df["visible"]),
    "stale_visible",
    "not_priority"
)

df["action"] = np.where(
    (df["stale"]) & (df["visible"]),
    "refresh_first",
    "monitor"
)

print("\nRule candidates:", (df["baseline_score"] > 0).sum())

top_candidates = df[df["baseline_score"] > 0].sort_values(
    "baseline_score",
    ascending=False
)

print("Number of candidates:", len(top_candidates))

top_candidates[
    [
        "content_id",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.80 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/flyrank-ml-internship-starter
Rows: 30000
Columns: 44
Staleness signal
   stale      n
0  False  29826
1   True    174

Visibility signal
   visible      n
0     True  16726
1    False  13274

Rule candidates: 17
Number of candidates: 17


,content_id,days_since_last_update,impressions_90d,avg_position,ctr,baseline_score,reason_code,action
16751,content_cf56e2e2e282,194,61678,19.7,0.15,61678,stale_visible,refresh_first
16514,content_7368877ea310,194,59472,24.8,0.13,59472,stale_visible,refresh_first
7021,content_1bfaa38ff26c,194,25715,22.2,0.23,25715,stale_visible,refresh_first
21268,content_0a91db491d14,193,13299,10.5,0.49,13299,stale_visible,refresh_first
11489,content_5feee3994adb,194,7812,39.0,0.01,7812,stale_visible,refresh_first
12045,content_c2d929d83eaa,193,7558,17.9,0.20,7558,stale_visible,refresh_first
698,content_b16bd7307b39,194,4590,31.0,0.00,4590,stale_visible,refresh_first
5327,content_fe16a55cd13d,194,4556,16.4,0.33,4556,stale_visible,refresh_first
26810,content_ecb6215e79fd,194,4429,25.3,0.38,4429,stale_visible,refresh_first
20837,content_928af3e22c80,193,1697,15.8,0.12,1697,stale_visible,refresh_first


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Build the Ranked Queue

The baseline ranks pages by the rule score. Pages that are both stale and visible receive a positive score based on their 90-day impressions. Pages that do not meet both conditions receive a score of zero.

The queue contains the content identifier, score, reason code, action, and the two signals used by the rule.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

queue = df.sort_values(
    ["baseline_score", "impressions_90d"],
    ascending=[False, False]
).copy()

queue["rank"] = range(1, len(queue) + 1)

queue_output = queue[
    [
        "rank",
        "content_id",
        "baseline_score",
        "reason_code",
        "action",
        "days_since_last_update",
        "impressions_90d",
        "avg_position",
        "ctr"
    ]
].copy()

os.makedirs("work/outputs", exist_ok=True)

queue_output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("Saved:", "work/outputs/baseline_action_score.csv")
print("Rows:", len(queue_output))

Saved: work/outputs/baseline_action_score.csv
Rows: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

The baseline rule produced 17 qualifying pages, so there are 17 candidates to review rather than 20. I do not add non-qualifying pages just to reach 20.

All 17 candidates receive the `refresh_first` action because they are both stale (at least 180 days since the last update) and visible (at least 500 impressions in the last 90 days).

The confidence note is limited because the rule uses only staleness and search exposure. A page may be stale and visible but still not need a refresh. A stronger review would also consider search intent, content quality, recent performance, and other signals.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top17 = queue_output[queue_output["baseline_score"] > 0].head(17).copy()

top17["confidence_note"] = (
    "Moderate confidence: stale and visible under the baseline rule."
)

top17["what_would_make_it_wrong"] = (
    "The page may already satisfy search intent or may not benefit from a refresh."
)

top17[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,rank,content_id,action,reason_code,confidence_note,what_would_make_it_wrong
16751,1,content_cf56e2e2e282,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
16514,2,content_7368877ea310,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
7021,3,content_1bfaa38ff26c,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
21268,4,content_0a91db491d14,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
11489,5,content_5feee3994adb,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
12045,6,content_c2d929d83eaa,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
698,7,content_b16bd7307b39,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
5327,8,content_fe16a55cd13d,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
26810,9,content_ecb6215e79fd,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...
20837,10,content_928af3e22c80,refresh_first,stale_visible,Moderate confidence: stale and visible under t...,The page may already satisfy search intent or ...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks + Leakage Check

Some qualifying pages are weaker recommendations than others even though they satisfy the rule.

For example, `content_074ba6ead17b` has only 533 impressions in 90 days, so the evidence of meaningful search exposure is relatively limited compared with the highest-ranked pages. The rule still selects it because it crosses the 500-impression threshold.

The rule also does not know whether a page already satisfies search intent or whether a content change would improve its performance.

The baseline uses only `days_since_last_update` and `impressions_90d`. It does not use `trend_pct`, `trend_direction`, `is_declining_label`, or future-window outcomes.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_features = [
    "days_since_last_update",
    "impressions_90d"
]

leakage_fields = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

print("Baseline features:", baseline_features)
print("Leakage fields used:", [
    col for col in leakage_fields
    if col in baseline_features
])

assert not any(
    col in baseline_features
    for col in leakage_fields
)

print("Leakage check: PASSED")

Baseline features: ['days_since_last_update', 'impressions_90d']
Leakage fields used: []
Leakage check: PASSED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.